#  Toyota Stock Price Prediction — Random Forest (Lag Features)

**Dataset:** `Toyota_Stock_Prices_1980_2026.csv`  
**Task:** Regression dengan Lag Features  
**Model:** Random Forest Regressor

---
###  Daftar Isi
1. Cara Melihat Tipe Data
2. Dataset Bisa Digunakan Untuk Apa
3. Model Yang Bisa Digunakan
4. Parameter Yang Bisa Diubah/Disetel
5. Evaluasi Yang Dipakai
6. Cara Mengetahui Evaluasi Bagus atau Tidak
7. Cara Mengoptimasi Model
8. Cara Menyimpan Model
9. Cara Menggunakan Model Hasil Training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print('Libraries loaded ')

---
## 1.  Cara Melihat Tipe Data

In [ ]:
df = pd.read_csv('../Toyota_Stock_Prices_1980_2026.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(df.shape); df.info()
df.head()

---
## 2.  Dataset Bisa Digunakan Untuk Apa

**Pendekatan ML Klasik untuk Time Series:**

Berbeda dari LSTM/ARIMA, kita konversi masalah time series menjadi supervised learning menggunakan **lag features**:

| Fitur | Keterangan |
|-------|------------|
| `close_lag_1` | Harga hari kemarin |
| `close_lag_5` | Harga 5 hari lalu |
| `ma_7` | Moving average 7 hari |
| `ma_30` | Moving average 30 hari |
| `volatility_7` | Std harga 7 hari terakhir |
| `day_of_week` | Hari dalam minggu (0=Senin..4=Jumat) |
| `month` | Bulan |

Target: **Harga Close besok**

In [ ]:
def create_lag_features(df_in, target_col='Close', lags=[1,2,3,5,10,20,30]):
    df_f = df_in.copy()
    
    # Lag features
    for lag in lags:
        df_f[f'close_lag_{lag}'] = df_f[target_col].shift(lag)
    
    # Moving averages
    df_f['ma_7']  = df_f[target_col].rolling(7).mean()
    df_f['ma_30'] = df_f[target_col].rolling(30).mean()
    df_f['ma_90'] = df_f[target_col].rolling(90).mean()
    
    # Volatility
    df_f['vol_7']  = df_f[target_col].rolling(7).std()
    df_f['vol_30'] = df_f[target_col].rolling(30).std()
    
    # Momentum
    df_f['momentum_5']  = df_f[target_col] - df_f[target_col].shift(5)
    df_f['momentum_20'] = df_f[target_col] - df_f[target_col].shift(20)
    
    # Price range
    df_f['daily_range'] = df_f['High'] - df_f['Low']
    df_f['range_5'] = df_f['daily_range'].rolling(5).mean()
    
    # Time features
    df_f['day_of_week'] = df_f['Date'].dt.dayofweek
    df_f['month']       = df_f['Date'].dt.month
    df_f['quarter']     = df_f['Date'].dt.quarter
    
    # Volume features
    df_f['vol_change'] = df_f['Volume'].pct_change()
    df_f['vol_ma_5']   = df_f['Volume'].rolling(5).mean()
    
    # Target: prediksi harga besok
    df_f['target'] = df_f[target_col].shift(-1)
    
    return df_f.dropna()

df_feat = create_lag_features(df[df['Date'] >= '2010-01-01'].copy())
print(f'Dataset setelah feature engineering: {df_feat.shape}')
print('\nFitur baru:', [c for c in df_feat.columns if c not in df.columns])

In [ ]:
feature_cols = [c for c in df_feat.columns if c not in ['Date', 'target', 'Close', 'Open', 'High', 'Low', 'Volume']]
X = df_feat[feature_cols].values
y = df_feat['target'].values

# Split kronologis 80/20
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## 3.  Kenapa Random Forest untuk Stock?

| Aspek | Random Forest | ARIMA/LSTM |
|-------|--------------|------------|
| Interpretasi fitur |  Feature importance | - |
| Kecepatan |  Sangat cepat | Lambat |
| Multivariate |  Mudah | Perlu modifikasi |
| Pattern non-linear |  Ya | LSTM ya, ARIMA tidak |
| Extrapolation |  Tidak bisa prediksi di luar range | ARIMA bisa |

**Limitasi Random Forest untuk time series:**
- Tidak bisa extrapolate (hanya interpolasi dalam range training)
- Tanpa lag features eksplisit, tidak menangkap temporal dependencies

In [ ]:
model = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=5,
    min_samples_leaf=2, max_features=0.7, oob_score=True,
    n_jobs=-1, random_state=42
)
model.fit(X_train, y_train)
print(f'Training selesai ')
print(f'OOB R²: {model.oob_score_:.4f}')

---
## 4.  Parameter Yang Bisa Diubah / Disetel

| Parameter | Default | Penjelasan |
|-----------|---------|------------|
| `n_estimators` | 100 | Jumlah pohon |
| `max_depth` | None | Kedalaman max pohon |
| `min_samples_split` | 2 | Min sampel untuk split node |
| `min_samples_leaf` | 1 | Min sampel di leaf |
| `max_features` | `'sqrt'` | Proporsi fitur per pohon |
| `oob_score` | False | Gunakan out-of-bag score |
| `n_jobs` | 1 | Paralelisme (−1 = semua core) |

---
## 5.  Evaluasi Yang Dipakai

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100
dir_acc = np.mean(np.sign(np.diff(y_test)) == np.sign(np.diff(y_pred))) * 100

print('='*55)
print(' EVALUASI RANDOM FOREST — STOCK PREDICTION')
print('='*55)
print(f'MAE            : {mae:.4f}')
print(f'RMSE           : {rmse:.4f}')
print(f'R²             : {r2:.4f}')
print(f'MAPE           : {mape:.2f}%')
print(f'Directional Acc: {dir_acc:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(y_test[:200], label='Aktual', color='steelblue')
axes[0].plot(y_pred[:200], label='RF Pred', color='tomato', alpha=0.8)
axes[0].set_title('Aktual vs RF Prediction (200 hari pertama test)')
axes[0].legend()

feat_imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(10)
feat_imp.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 10 Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout(); plt.show()

---
## 6.  Cara Mengetahui Evaluasi Bagus atau Tidak

###  Peringatan Look-Ahead Bias!

Salah satu masalah umum dalam prediksi saham:
- Pastikan **hanya menggunakan data T-1 dan sebelumnya** untuk prediksi T
- Jangan ada feature yang menggunakan informasi masa depan
- Validasi selalu harus kronologis (bukan random)

### Benchmark:
- **Baseline** terbaik: prediksi hari ini sama dengan kemarin (naive model)
- Model yang baik harus **lebih baik dari baseline**

```python
# Naive baseline: prediksi = harga kemarin
naive_pred = X_test[:, feature_cols.index('close_lag_1')]
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
print(f'Naive RMSE: {naive_rmse:.4f}')
print(f'RF RMSE   : {rmse:.4f}')
```

In [ ]:
# Bandingkan dengan naive baseline
lag1_idx = list(feature_cols).index('close_lag_1')
naive_pred = X_test[:, lag1_idx]
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
naive_mape = np.mean(np.abs((y_test - naive_pred) / (y_test + 1e-8))) * 100

print('Perbandingan dengan Naive Baseline:')
print(f'  Naive RMSE: {naive_rmse:.4f}, MAPE: {naive_mape:.2f}%')
print(f'  RF RMSE   : {rmse:.4f}, MAPE: {mape:.2f}%')
print(f'  Improvement: RMSE {(naive_rmse - rmse)/naive_rmse*100:.1f}%, MAPE {(naive_mape - mape)/naive_mape*100:.1f}%')

---
## 7.  Cara Mengoptimasi Model

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.5, 0.7, 'sqrt']
}

rs = RandomizedSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=42),
    param_dist, n_iter=10, cv=5, scoring='r2',
    random_state=42, n_jobs=-1
)
rs.fit(X_train, y_train)
best_model = rs.best_estimator_
y_best = best_model.predict(X_test)

print(f'Best params: {rs.best_params_}')
print(f'Best R²  : {r2_score(y_test, y_best):.4f}')
print(f'Best RMSE: {np.sqrt(mean_squared_error(y_test, y_best)):.4f}')

---
## 8.  Cara Menyimpan Model

In [ ]:
os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_model, 'saved_models/rf_stock.pkl')
joblib.dump(feature_cols, 'saved_models/feature_cols_rf_stock.pkl')
print(' RF Stock model tersimpan!')

---
## 9.  Cara Menggunakan Model Hasil Training

In [ ]:
loaded_model = joblib.load('saved_models/rf_stock.pkl')
loaded_cols = joblib.load('saved_models/feature_cols_rf_stock.pkl')
print('Model dimuat ')

# Prediksi berdasarkan data terakhir
last_row = df_feat.tail(1)[loaded_cols]
pred_price = loaded_model.predict(last_row)[0]
last_price = df_feat['Close'].iloc[-1]

print(f'\nHarga Close terakhir   : ${last_price:.2f}')
print(f'Prediksi harga besok   : ${pred_price:.2f}')
print(f'Prediksi perubahan     : {((pred_price - last_price) / last_price * 100):+.2f}%')
direction = ' NAIK' if pred_price > last_price else ' TURUN'
print(f'Arah prediksi          : {direction}')